In [40]:
import pandas as pd
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 120) 
pd.set_option("display.max_columns", 100)

In [6]:
df_hsv = pd.read_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/features/hsv_features.csv", skipinitialspace=True) 
df_mhi = pd.read_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/features/mhi_features.csv", skipinitialspace=True) 
df_glcm = pd.read_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/features/glcm_features.csv", skipinitialspace=True) 
df_flow = pd.read_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/features/flow_features.csv", skipinitialspace=True) 
df_hog = pd.read_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/features/hog_features.csv", skipinitialspace=True) 
df_metadata = pd.read_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/features/metadata.csv", skipinitialspace=True) 

In [10]:
for df in [df_mhi, df_metadata, df_flow, df_glcm, df_hsv, df_hog]: 
    df.columns = df.columns.str.strip() 

In [13]:
def normalize_video_name(value):
    if pd.isna(value):
        return value

    return str(value).replace(
        "dog_eat_drink_",
        "dog_eat_"
    )

In [16]:
df_mhi["path"] = df_mhi["path"].apply(normalize_video_name) 
df_hog["path"] = df_hog["path"].apply(normalize_video_name) 
df_flow["path"] = df_flow["path"].apply(normalize_video_name) 

In [18]:
for df in [df_mhi, df_metadata, df_flow, df_glcm, df_hsv, df_hog]: 
    df.columns = df.columns.str.strip() 

In [24]:
df_mhi["path"][200] 

'/Users/trantuanduong/Documents/CNTT_PTIT/Học kỳ 7/HCSDLĐPT/data/eat_drink/dog_eat_14.mp4  '

In [25]:
import os 
def get_filename(path):
    if pd.isna(path):
        return None

    path = str(path).strip()

    # Chuẩn hóa dấu phân cách đường dẫn
    path = path.replace("\\", "/")

    return os.path.basename(path)

In [26]:
df_metadata["filename"] = (
    df_metadata["filename"]
    .astype(str)
    .str.strip()
)

df_metadata["filepath"] = (
    df_metadata["filepath"]
    .astype(str)
    .str.strip()
)

df_metadata["filename"] = df_metadata["filename"].apply(get_filename)

In [27]:
df_hsv["filename"] = df_hsv["video_path"].apply(get_filename)

df_glcm["filename"] = df_glcm["video_path"].apply(get_filename)

df_flow["filename"] = df_flow["path"].apply(get_filename)

df_mhi["filename"] = df_mhi["path"].apply(get_filename)

df_hog["filename"] = df_hog["path"].apply(get_filename)

In [29]:
# Tạo ID mới từ metadata
df_metadata = df_metadata.reset_index(drop=True)

df_metadata.insert(
    0,
    "video_id",
    range(1, len(df_metadata) + 1)
)

In [33]:
manual_mapping = pd.read_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/features/manual_mapping.csv")

manual_mapping.columns = manual_mapping.columns.str.strip()

manual_dict = dict(
    zip(
        manual_mapping["old_filename"],
        manual_mapping["new_filename"]
    )
)

In [34]:
df_flow["filename"] = df_flow["filename"].replace(manual_dict) 
df_mhi["filename"] = df_mhi["filename"].replace(manual_dict) 
df_hog["filename"] = df_hog["filename"].replace(manual_dict) 

In [ ]:
video_mapping = df_metadata[
    ["video_id", "filename", "filepath"]
].copy() 

video_mapping.head(66)

,video_id,filename,filepath
0,1,dog_eat_1.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4
1,2,dog_eat_20.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_20.mp4
2,3,dog_eat_66.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_66.mp4
3,4,dog_eat_54.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_54.mp4
4,5,dog_eat_59.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_59.mp4
...,...,...,...
61,62,dog_eat_39.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_39.mp4
62,63,dog_eat_51.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_51.mp4
63,64,dog_eat_69.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_69.mp4
64,65,dog_eat_72.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_72.mp4


In [44]:
test_df_hsv = df_hsv.merge(
    video_mapping,
    on="filename",
    how="left",
    validate="many_to_one"
)

test_df_hsv = test_df_hsv.rename(
    columns={
        "timestamp": "timestamp_sec",
        "hsv_vector": "vector"
    }
)

test_df_hsv = test_df_hsv[
    [
        "video_id",
        "filepath",
        "frame_idx",
        "timestamp_sec",
        "vector"
    ]
]

test_df_hsv 

,video_id,filepath,frame_idx,timestamp_sec,vector
0,1.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,0,0.000000,"[0.02617924528301887, 0.15947572720125786, 0.14300560141509433, 0.004999508647798742, 0.0035598466981132076, 1.47405..."
1,1.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,30,1.002124,"[0.02809060534591195, 0.15074685534591195, 0.1630282036163522, 0.004456564465408805, 0.002793337264150943, 0.0, 0.00..."
2,1.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,60,2.004249,"[0.033814858490566035, 0.15899174528301888, 0.14856033805031446, 0.004925805817610063, 0.0030169025157232706, 3.1937..."
3,1.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,90,3.006373,"[0.029247739779874212, 0.17289946933962264, 0.1443224253144654, 0.00582498034591195, 0.0022675904088050314, 9.335691..."
4,1.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,120,4.008498,"[0.031257370283018865, 0.1859571540880503, 0.14530267295597485, 0.0070263364779874216, 0.005761104559748428, 0.00014..."
...,...,...,...,...,...
5918,504.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,160,5.066667,"[0.03300010175010175, 0.1827498600936101, 0.09560407763532763, 0.014410994098494098, 0.013188721001221001, 0.0005135..."
5919,504.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,192,6.080000,"[0.027061075498575497, 0.16922854344729343, 0.08970765923890923, 0.013386497761497761, 0.013176956145706145, 0.00028..."
5920,504.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,224,7.093333,"[0.0256044591982092, 0.16512197293447292, 0.08431712962962963, 0.013245637464387464, 0.011409366096866097, 0.0003100..."
5921,504.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,256,8.106667,"[0.026929118335368336, 0.15871267043142043, 0.0885423026048026, 0.012749923687423687, 0.011292989417989417, 0.000361..."


In [47]:
test_df_glcm = df_glcm.drop(columns=["video_id"]).merge(
    video_mapping,
    on="filename",
    how="left",
    validate="many_to_one"
)

test_df_glcm = test_df_glcm.rename(
    columns={
        "glcm_vector": "vector"
    }
)

test_df_glcm = test_df_glcm[
    [
        "video_id",
        "filepath",
        "frame_idx",
        "timestamp_sec",
        "vector"
    ]
]
test_df_glcm

,video_id,filepath,frame_idx,timestamp_sec,vector
0,1.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,0,0.000,"[0.11328129470348358, 0.31043195724487305, 3.679398536682129, 0.8967195749282837]"
1,1.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,30,1.002,"[0.10935035347938538, 0.31993210315704346, 3.7216877937316895, 0.8939757943153381]"
2,1.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,60,2.004,"[0.10772231966257095, 0.3149460256099701, 3.7200679779052734, 0.8960456252098083]"
3,1.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,90,3.006,"[0.1101420670747757, 0.2990172505378723, 3.6747984886169434, 0.8996459245681763]"
4,1.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,120,4.008,"[0.10790690034627914, 0.33149415254592896, 3.7249674797058105, 0.891003429889679]"
...,...,...,...,...,...
5918,504.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,160,5.067,"[0.13164307177066803, 0.2307320535182953, 3.4436392784118652, 0.9050720930099487]"
5919,504.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,192,6.080,"[0.13610300421714783, 0.2032986581325531, 3.379974365234375, 0.9137535691261292]"
5920,504.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,224,7.093,"[0.13256525993347168, 0.2116389274597168, 3.4144790172576904, 0.9104992151260376]"
5921,504.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,256,8.107,"[0.13607095181941986, 0.1933104395866394, 3.3655920028686523, 0.9156087636947632]"


In [48]:
test_df_flow = df_flow.drop(columns=["video_id"]).merge(
    video_mapping,
    on="filename",
    how="left",
    validate="many_to_one"
)

test_df_flow = test_df_flow.rename(
    columns={
        "optical_flow_vector": "vector"
    }
)

test_df_flow = test_df_flow[
    [
        "video_id",
        "filepath",
        "vector"
    ]
]
test_df_flow

,video_id,filepath,vector
0,31.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_4.mp4,"[0.1894336640834808, 0.0538028478622436, 0.0538028478622436, 0.1894336640834808, 0.2074091434478759, 0.0493543334305..."
1,66.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_2.mp4,"[0.1529983878135681, 0.0856940448284149, 0.0856940448284149, 0.1529983878135681, 0.1676016598939895, 0.0937059000134..."
2,1.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,"[0.2318705320358276, 0.0241991709917783, 0.0241991709917783, 0.2318705320358276, 0.1829137951135635, 0.0610164999961..."
3,48.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_3.mp4,"[0.1781081855297088, 0.071504071354866, 0.071504071354866, 0.1781081855297088, 0.1882883012294769, 0.062099438160657..."
4,7.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_5.mp4,"[0.1747577488422393, 0.0483232736587524, 0.0483232736587524, 0.1747577488422393, 0.1945402026176452, 0.0823787823319..."
...,...,...,...
497,135.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_73.mp4,"[0.0051622251048684, 0.002950916532427, 0.002950916532427, 0.0051622251048684, 0.3577311635017395, 0.134155690670013..."
498,90.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_76.mp4,"[0.1553722918033599, 0.0869016274809837, 0.0869016274809837, 0.1553722918033599, 0.2091047912836074, 0.0486212819814..."
499,75.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_71.mp4,"[0.2472555935382843, 0.2010935395956039, 0.2010935395956039, 0.2472555935382843, 0.0373349338769912, 0.0143159236758..."
500,124.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_77.mp4,"[0.1587321460247039, 0.0049533485434949, 0.0049533485434949, 0.1587321460247039, 0.3298554122447967, 0.0064590908586..."


In [50]:
# Đồng bộ với metadata
test_df_mhi = df_mhi.drop(columns=["video_id"]).merge(
    video_mapping,
    on="filename",
    how="left",
    validate="many_to_one"
)

test_df_mhi = test_df_mhi.rename(
    columns={
        "mhi_vector": "vector"
    }
)

test_df_mhi = test_df_mhi[
    [
        "video_id",
        "filepath",
        "mhi_window_index",
        "vector"
    ]
]

test_df_mhi 

,video_id,filepath,mhi_window_index,vector
0,31.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_4.mp4,0,"[0.0, 0.0533333346247673, 0.4250000119209289, 0.4875000119209289, 0.2191666662693023, 0.0608333349227905, 0.06500000..."
1,31.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_4.mp4,1,"[0.0, 0.0, 0.0766666680574417, 0.2600000202655792, 0.3041666746139526, 0.1108333393931388, 0.0041666668839752, 0.063..."
2,31.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_4.mp4,2,"[0.0, 0.0, 0.0199999995529651, 0.2425000071525573, 0.2083333432674408, 0.1599999964237213, 0.0983333364129066, 0.168..."
3,31.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_4.mp4,3,"[0.7200000286102295, 0.9991666674613952, 0.912500023841858, 0.8075000047683716, 0.9133333563804626, 0.84750002622604..."
4,31.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_4.mp4,4,"[0.5550000071525574, 0.4783333539962768, 0.4408333301544189, 0.4558333456516266, 0.940833330154419, 0.54583334922790..."
...,...,...,...,...
5753,146.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_69.mp4,14,"[0.2199999988079071, 0.4691666662693023, 0.2341666668653488, 0.3516666889190674, 0.5074999928474426, 0.6058333516120..."
5754,146.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_69.mp4,15,"[0.1416666656732559, 0.5, 0.4141666889190674, 0.1133333370089531, 0.6075000166893005, 0.6675000190734863, 0.50499999..."
5755,146.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_69.mp4,16,"[0.1816666722297668, 0.4208333492279053, 0.5041666626930237, 0.1533333361148834, 0.51583331823349, 0.741666674613952..."
5756,146.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_69.mp4,17,"[0.2400000095367431, 0.2183333337306976, 0.5200000405311584, 0.1875, 0.2041666656732559, 0.7091667056083679, 0.53750..."


In [52]:
test_df_hog =  df_hog.drop(columns=["video_id"]).merge(
    video_mapping,
    on="filename",
    how="left",
    validate="many_to_one"
)

test_df_hog = test_df_hog.rename(
    columns={
        "timestamp_seconds": "timestamp_sec",
        "hog_vector": "vector", 
        "frame_index": "frame_idx"
    }
)

test_df_hog = test_df_hog[
    [
        "video_id",
        "filepath",
        "frame_idx",
        "timestamp_sec",
        "vector"
    ]
]

test_df_hog

,video_id,filepath,frame_idx,timestamp_sec,vector
0,66.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_2.mp4,210,7.033,"[0.255672216, 0.00923696719, 0.155310899, 0.0879012421, 0.255672216, 0.116643161, 0.0913366079, 0.0184739344, 0.0130..."
1,66.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_2.mp4,300,10.033,"[0.188499227, 0.110608034, 0.191171497, 0.0308265407, 0.263885021, 0.0361780003, 0.045761954, 0.0, 0.0898128524, 0.2..."
2,66.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_2.mp4,270,9.033,"[0.254954696, 0.114168927, 0.122684762, 0.0523081198, 0.254954696, 0.102181047, 0.0732445419, 0.0275568683, 0.025980..."
3,66.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_2.mp4,240,8.033,"[0.254163653, 0.0619932301, 0.129638433, 0.155053109, 0.258765787, 0.0464297459, 0.0366847031, 0.0540696234, 0.02800..."
4,66.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_2.mp4,390,13.033,"[0.253879637, 0.0468086973, 0.186102733, 0.170732468, 0.253879637, 0.180461884, 0.0283560026, 0.0492541417, 0.032330..."
...,...,...,...,...,...
5863,146.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_69.mp4,390,13.013,"[0.279446095, 0.0871039182, 0.099561438, 0.0551614389, 0.220911667, 0.126329765, 0.226265594, 0.279446095, 0.2794460..."
5864,146.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_69.mp4,450,15.015,"[0.227722391, 0.0447220393, 0.0336435772, 0.039614141, 0.110185362, 0.121464871, 0.0484782681, 0.139078319, 0.094719..."
5865,146.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_69.mp4,480,16.016,"[0.199885026, 0.00343724433, 0.0141721219, 0.0474572703, 0.0697076619, 0.0585389547, 0.0803257525, 0.123578683, 0.17..."
5866,146.0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_69.mp4,540,18.018,"[0.316912442, 0.183315217, 0.104783319, 0.0839906111, 0.194007412, 0.0711794198, 0.106145568, 0.198710695, 0.0980438..."


In [53]:
keyframes = pd.concat(
    [
        test_df_hsv[
            ["video_id", "frame_idx", "timestamp_sec"]
        ],
        test_df_glcm[
            ["video_id", "frame_idx", "timestamp_sec"]
        ],
        test_df_hog[
            ["video_id", "frame_idx", "timestamp_sec"]
        ]
    ],
    ignore_index=True
)

# Loại bỏ các keyframe trùng nhau
keyframes = keyframes.drop_duplicates(
    subset=["video_id", "frame_idx"]
)

# Sắp xếp
keyframes = keyframes.sort_values(
    ["video_id", "frame_idx"]
).reset_index(drop=True)

# Tạo keyframe_id
keyframes.insert(
    0,
    "keyframe_id",
    range(1, len(keyframes) + 1)
)

# Tạo keyframe_index theo từng video
keyframes["keyframe_index"] = (
    keyframes.groupby("video_id").cumcount()
)

keyframes 

,keyframe_id,video_id,frame_idx,timestamp_sec,keyframe_index
0,1,1.0,0,0.000000,0.0
1,2,1.0,30,1.002124,1.0
2,3,1.0,60,2.004249,2.0
3,4,1.0,90,3.006373,3.0
4,5,1.0,120,4.008498,4.0
...,...,...,...,...,...
6472,6473,NaN,1223,51.009292,NaN
6473,6474,NaN,1247,52.010292,NaN
6474,6475,NaN,1271,53.011292,NaN
6475,6476,NaN,1295,54.012292,NaN


In [54]:
test_df_hsv = test_df_hsv.merge(
    keyframes[
        ["keyframe_id", "video_id", "frame_idx"]
    ],
    on=["video_id", "frame_idx"],
    how="left",
    validate="one_to_one"
)

test_df_hsv = test_df_hsv[
    ["keyframe_id", "vector"]
]

test_df_hsv 

,keyframe_id,vector
0,1,"[0.02617924528301887, 0.15947572720125786, 0.14300560141509433, 0.004999508647798742, 0.0035598466981132076, 1.47405..."
1,2,"[0.02809060534591195, 0.15074685534591195, 0.1630282036163522, 0.004456564465408805, 0.002793337264150943, 0.0, 0.00..."
2,3,"[0.033814858490566035, 0.15899174528301888, 0.14856033805031446, 0.004925805817610063, 0.0030169025157232706, 3.1937..."
3,4,"[0.029247739779874212, 0.17289946933962264, 0.1443224253144654, 0.00582498034591195, 0.0022675904088050314, 9.335691..."
4,5,"[0.031257370283018865, 0.1859571540880503, 0.14530267295597485, 0.0070263364779874216, 0.005761104559748428, 0.00014..."
...,...,...
5918,6373,"[0.03300010175010175, 0.1827498600936101, 0.09560407763532763, 0.014410994098494098, 0.013188721001221001, 0.0005135..."
5919,6375,"[0.027061075498575497, 0.16922854344729343, 0.08970765923890923, 0.013386497761497761, 0.013176956145706145, 0.00028..."
5920,6377,"[0.0256044591982092, 0.16512197293447292, 0.08431712962962963, 0.013245637464387464, 0.011409366096866097, 0.0003100..."
5921,6379,"[0.026929118335368336, 0.15871267043142043, 0.0885423026048026, 0.012749923687423687, 0.011292989417989417, 0.000361..."


In [ ]:
test_df_glcm = test_df_glcm.merge(
    keyframes[
        ["keyframe_id", "video_id", "frame_idx"]
    ],
    on=["video_id", "frame_idx"],
    how="left",
    validate="one_to_one"
)

test_df_glcm = test_df_glcm[
    ["keyframe_id", "vector"]
]

KeyError: 'video_id'

In [57]:
test_df_glcm

,keyframe_id,vector
0,1,"[0.11328129470348358, 0.31043195724487305, 3.679398536682129, 0.8967195749282837]"
1,2,"[0.10935035347938538, 0.31993210315704346, 3.7216877937316895, 0.8939757943153381]"
2,3,"[0.10772231966257095, 0.3149460256099701, 3.7200679779052734, 0.8960456252098083]"
3,4,"[0.1101420670747757, 0.2990172505378723, 3.6747984886169434, 0.8996459245681763]"
4,5,"[0.10790690034627914, 0.33149415254592896, 3.7249674797058105, 0.891003429889679]"
...,...,...
5918,6373,"[0.13164307177066803, 0.2307320535182953, 3.4436392784118652, 0.9050720930099487]"
5919,6375,"[0.13610300421714783, 0.2032986581325531, 3.379974365234375, 0.9137535691261292]"
5920,6377,"[0.13256525993347168, 0.2116389274597168, 3.4144790172576904, 0.9104992151260376]"
5921,6379,"[0.13607095181941986, 0.1933104395866394, 3.3655920028686523, 0.9156087636947632]"


In [58]:
test_df_hog = test_df_hog.merge(
    keyframes[
        ["keyframe_id", "video_id", "frame_idx"]
    ],
    on=["video_id", "frame_idx"],
    how="left",
    validate="one_to_one"
)

test_df_hog = test_df_hog[
    ["keyframe_id", "vector"]
]

test_df_hog

MergeError: Merge keys are not unique in left dataset; not a one-to-one merge

In [41]:
print("="*50 + "METADATA" + "="*50) 
display(df_metadata.head()) 
print("="*50 + "HSV" + "="*50) 
display(df_hsv.head()) 
print("="*50 + "GLCM" + "="*50) 
display(df_glcm.head()) 
print("="*50 + "MHI" + "="*50) 
display(df_mhi.head(10)) 
print("="*50 + "FLOW" + "="*50) 
display(df_flow.head(10)) 
print("="*50 + "HOG" + "="*50) 
display(df_hog.head(10)) 

==================================================METADATA==================================================


,video_id,uuid,filename,filepath,category,fps,total_frames,duration,width,height
0,1,dog_eat_1,dog_eat_1.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,Eating,29.936401,466,15.566333,848,480
1,2,dog_eat_20,dog_eat_20.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_20.mp4,Eating,24.000000,129,5.375000,720,1280
2,3,dog_eat_66,dog_eat_66.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_66.mp4,Eating,24.000000,325,13.541667,2732,1440
3,4,dog_eat_54,dog_eat_54.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_54.mp4,Eating,60.000000,1124,18.733333,2160,3840
4,5,dog_eat_59,dog_eat_59.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_59.mp4,Eating,25.000000,293,11.720000,1920,1080


==================================================HSV==================================================


,video_path,frame_idx,timestamp,hsv_vector,filename
0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,0,0.000000,"[0.02617924528301887, 0.15947572720125786, 0.14300560141509433, 0.004999508647798742, 0.0035598466981132076, 1.47405...",dog_eat_1.mp4
1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,30,1.002124,"[0.02809060534591195, 0.15074685534591195, 0.1630282036163522, 0.004456564465408805, 0.002793337264150943, 0.0, 0.00...",dog_eat_1.mp4
2,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,60,2.004249,"[0.033814858490566035, 0.15899174528301888, 0.14856033805031446, 0.004925805817610063, 0.0030169025157232706, 3.1937...",dog_eat_1.mp4
3,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,90,3.006373,"[0.029247739779874212, 0.17289946933962264, 0.1443224253144654, 0.00582498034591195, 0.0022675904088050314, 9.335691...",dog_eat_1.mp4
4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,120,4.008498,"[0.031257370283018865, 0.1859571540880503, 0.14530267295597485, 0.0070263364779874216, 0.005761104559748428, 0.00014...",dog_eat_1.mp4


==================================================GLCM==================================================


,video_id,video_path,video_name,frame_idx,timestamp_sec,glcm_vector,filename
0,0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,dog_eat_1.mp4,0,0.000,"[0.11328129470348358, 0.31043195724487305, 3.679398536682129, 0.8967195749282837]",dog_eat_1.mp4
1,0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,dog_eat_1.mp4,30,1.002,"[0.10935035347938538, 0.31993210315704346, 3.7216877937316895, 0.8939757943153381]",dog_eat_1.mp4
2,0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,dog_eat_1.mp4,60,2.004,"[0.10772231966257095, 0.3149460256099701, 3.7200679779052734, 0.8960456252098083]",dog_eat_1.mp4
3,0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,dog_eat_1.mp4,90,3.006,"[0.1101420670747757, 0.2990172505378723, 3.6747984886169434, 0.8996459245681763]",dog_eat_1.mp4
4,0,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,dog_eat_1.mp4,120,4.008,"[0.10790690034627914, 0.33149415254592896, 3.7249674797058105, 0.891003429889679]",dog_eat_1.mp4


... (2 outputs left out)

==================================================FLOW==================================================


,video_id,path,label,optical_flow_vector,filename
0,1,/Users/trantuanduong/Documents/CNTT_PTIT/Học kỳ 7/HCSDLĐPT/data/eat_drink/dog_eat_04.mp4,eat_drink,"[0.1894336640834808, 0.0538028478622436, 0.0538028478622436, 0.1894336640834808, 0.2074091434478759, 0.0493543334305...",dog_eat_4.mp4
1,2,/Users/trantuanduong/Documents/CNTT_PTIT/Học kỳ 7/HCSDLĐPT/data/eat_drink/dog_eat_02.mp4,eat_drink,"[0.1529983878135681, 0.0856940448284149, 0.0856940448284149, 0.1529983878135681, 0.1676016598939895, 0.0937059000134...",dog_eat_2.mp4
2,3,/Users/trantuanduong/Documents/CNTT_PTIT/Học kỳ 7/HCSDLĐPT/data/eat_drink/dog_eat_01.mp4,eat_drink,"[0.2318705320358276, 0.0241991709917783, 0.0241991709917783, 0.2318705320358276, 0.1829137951135635, 0.0610164999961...",dog_eat_1.mp4
3,4,/Users/trantuanduong/Documents/CNTT_PTIT/Học kỳ 7/HCSDLĐPT/data/eat_drink/dog_eat_03.mp4,eat_drink,"[0.1781081855297088, 0.071504071354866, 0.071504071354866, 0.1781081855297088, 0.1882883012294769, 0.062099438160657...",dog_eat_3.mp4
4,5,/Users/trantuanduong/Documents/CNTT_PTIT/Học kỳ 7/HCSDLĐPT/data/eat_drink/dog_eat_05.mp4,eat_drink,"[0.1747577488422393, 0.0483232736587524, 0.0483232736587524, 0.1747577488422393, 0.1945402026176452, 0.0823787823319...",dog_eat_5.mp4
5,6,/Users/trantuanduong/Documents/CNTT_PTIT/Học kỳ 7/HCSDLĐPT/data/eat_drink/dog_eat_06.mp4,eat_drink,"[0.213648185133934, 0.0688893646001815, 0.0688893646001815, 0.213648185133934, 0.1709841638803482, 0.046478286385536...",dog_eat_6.mp4
6,7,/Users/trantuanduong/Documents/CNTT_PTIT/Học kỳ 7/HCSDLĐPT/data/eat_drink/dog_eat_08.mp4,eat_drink,"[0.2227473258972168, 0.0584275014698505, 0.0584275014698505, 0.2227473258972168, 0.1477896124124527, 0.0710355713963...",dog_eat_8.mp4
7,8,/Users/trantuanduong/Documents/CNTT_PTIT/Học kỳ 7/HCSDLĐPT/data/eat_drink/dog_eat_07.mp4,eat_drink,"[0.2186099141836166, 0.0417207367718219, 0.0417207367718219, 0.2186099141836166, 0.20108263194561, 0.038586709648370...",dog_eat_7.mp4
8,9,/Users/trantuanduong/Documents/CNTT_PTIT/Học kỳ 7/HCSDLĐPT/data/eat_drink/dog_eat_09.mp4,eat_drink,"[0.231179267168045, 0.0484852716326713, 0.0484852716326713, 0.231179267168045, 0.1689554303884506, 0.051380027085542...",dog_eat_9.mp4
9,10,/Users/trantuanduong/Documents/CNTT_PTIT/Học kỳ 7/HCSDLĐPT/data/eat_drink/dog_eat_11.mp4,eat_drink,"[0.0907506197690963, 0.1339595317840576, 0.1339595317840576, 0.0907506197690963, 0.1147273257374763, 0.1605625152587...",dog_eat_11.mp4


==================================================HOG==================================================


,video_id,path,keyframe_index,timestamp_seconds,frame_index,hog_vector,filename
0,1,C:\Users\Admin\OneDrive\Pictures\Documents\Kì 7\CSDL ĐPT\eat_drink\dog_eat_02.mp4,7,7.033,210,"[0.255672216, 0.00923696719, 0.155310899, 0.0879012421, 0.255672216, 0.116643161, 0.0913366079, 0.0184739344, 0.0130...",dog_eat_2.mp4
1,1,C:\Users\Admin\OneDrive\Pictures\Documents\Kì 7\CSDL ĐPT\eat_drink\dog_eat_02.mp4,10,10.033,300,"[0.188499227, 0.110608034, 0.191171497, 0.0308265407, 0.263885021, 0.0361780003, 0.045761954, 0.0, 0.0898128524, 0.2...",dog_eat_2.mp4
2,1,C:\Users\Admin\OneDrive\Pictures\Documents\Kì 7\CSDL ĐPT\eat_drink\dog_eat_02.mp4,9,9.033,270,"[0.254954696, 0.114168927, 0.122684762, 0.0523081198, 0.254954696, 0.102181047, 0.0732445419, 0.0275568683, 0.025980...",dog_eat_2.mp4
3,1,C:\Users\Admin\OneDrive\Pictures\Documents\Kì 7\CSDL ĐPT\eat_drink\dog_eat_02.mp4,8,8.033,240,"[0.254163653, 0.0619932301, 0.129638433, 0.155053109, 0.258765787, 0.0464297459, 0.0366847031, 0.0540696234, 0.02800...",dog_eat_2.mp4
4,1,C:\Users\Admin\OneDrive\Pictures\Documents\Kì 7\CSDL ĐPT\eat_drink\dog_eat_02.mp4,13,13.033,390,"[0.253879637, 0.0468086973, 0.186102733, 0.170732468, 0.253879637, 0.180461884, 0.0283560026, 0.0492541417, 0.032330...",dog_eat_2.mp4
5,1,C:\Users\Admin\OneDrive\Pictures\Documents\Kì 7\CSDL ĐPT\eat_drink\dog_eat_02.mp4,6,6.033,180,"[0.214709938, 0.0749305189, 0.142528623, 0.127153397, 0.245647341, 0.190464914, 0.0, 0.0486852825, 0.0689985231, 0.2...",dog_eat_2.mp4
6,1,C:\Users\Admin\OneDrive\Pictures\Documents\Kì 7\CSDL ĐPT\eat_drink\dog_eat_02.mp4,5,5.033,150,"[0.243607491, 0.0993755385, 0.141869441, 0.0857305825, 0.252212733, 0.122548684, 0.0516030304, 0.0, 0.0411997885, 0....",dog_eat_2.mp4
7,1,C:\Users\Admin\OneDrive\Pictures\Documents\Kì 7\CSDL ĐPT\eat_drink\dog_eat_02.mp4,4,4.033,120,"[0.244185373, 0.134575129, 0.143706948, 0.0691043139, 0.244185373, 0.181652889, 0.0296497419, 0.00919399504, 0.02907...",dog_eat_2.mp4
8,1,C:\Users\Admin\OneDrive\Pictures\Documents\Kì 7\CSDL ĐPT\eat_drink\dog_eat_02.mp4,3,3.033,90,"[0.240425497, 0.0571707636, 0.10012649, 0.0849299803, 0.246057749, 0.210349813, 0.0650397539, 0.0233154427, 0.069189...",dog_eat_2.mp4
9,1,C:\Users\Admin\OneDrive\Pictures\Documents\Kì 7\CSDL ĐPT\eat_drink\dog_eat_02.mp4,2,2.033,60,"[0.192872062, 0.136698201, 0.120706186, 0.0568255894, 0.267341048, 0.093628712, 0.045137424, 0.00947093219, 0.056588...",dog_eat_2.mp4
